In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp Dây cung (Secant Method)

# Điều kiện:

* $(a,b)$ là khoảng cách ly nghiệm

* $f'(x)$ xác định dấu không đổi trên $[a,b]$

* $f''(x)$ xác định dấu không đổi trên $[a,b]$

Điều kiện hội tụ:

* Chọn đúng điểm mốc $d$ (điểm Fourier - $f(d) \cdot f''(d) > 0$)
    
* Chọn đúng xuất phát ban đầu $x_0$ ($f(d) \cdot f(x_0) < 0$)

# Thuật toán tổng quát:

1. Kiểm tra input ban đầu có thỏa mãn điều kiện không

2. Xác định điểm mốc $d$ và điểm xuất phát ban đầu $x_0$

3. Xác định x theo công thức lặp: 

$\quad x_{n+1} = x_n - \dfrac{f(x_n) \cdot (x_n - d)}{f(x_n) - f(d)} = \dfrac{d \cdot f(x_n) - x_n \cdot f(d)}{f(x_n) - f(d)}$

4. Lặp lại bước (3) cho đến khi:

   a. $x_n$ là nghiệm của phương trình, hoặc

   b. khoảng $[d, x_n]$ (hoặc $[x_n, d]$) thỏa mãn điều kiện dừng

5. In giá trị $x_n$ (giá trị $x$ sau $n$ lần lặp)

# Áp dụng

## 1. Hậu nghiệm
Đánh giá sai số theo 1 trong 2 công thức:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1}$ $\quad\quad$(1)

+ $|x_n - x^*| \leqslant \dfrac{M_1 - m_1}{m_1} \cdot |x_n - x_{n-1}| $ $\quad\quad$(2)

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_1 = \max\limits_{[a,b]} |f'(x)|$

### 1.1 Thuật toán
#### Input:
- Khoảng cách ly nghiệm $(a,b)$
- Sai số $\varepsilon$
- Hàm số $f(x)$ thoả mãn $f'(x), f''(x)$ xác định không đổi dấu trên $[a,b]$
- Giá trị nhỏ nhất ($m_1$) và giá trị lớn nhất ($M_1$) của $|f'(x)|$ trên $[a,b]$
#### Output: nghiệm gần đúng $x$
#### Thuật toán:
- Bước 1: Kiểm tra điều kiện tồn tại nghiệm:
    - Nếu $f(a) \cdot f(b) > 0$ hoặc $m_1 \cdot M_1 \leqslant 0$: dừng chương trình do không thoả mãn điều kiện đầu vào
    - Nếu $f(a)=0$: thu được nghiệm $x=a$, dừng chương trình
    - Nếu $f(b)=0$: thu được nghiệm $x=b$, dừng chương trình
- Bước 2: Chọn $x_0=a, \, d=b$
- Bước 3: Khởi tạo $i=0$. Lặp đến khi thoả mãn điều kiện dừng $|f(x_n)| < m_1 \varepsilon $ hoặc $|x_{n}-x_{n-1}| < \varepsilon\dfrac{m_1}{M_1-m_1}$:
    - Tính $x_{i+1}=\dfrac{d \cdot f(x_{i}) - x_{i+1} \cdot f(d)}{f(x_{i+1})-f(d)}, \quad \Delta x_{i}=\dfrac{|f(x_{i+1})|}{m_1}$ hoặc $\Delta x_{i}=\dfrac{M_1 - m_1}{m_1}\cdot |x_{i+1}-x_{i}|$
    - Nếu $f(x_{i+1})=0$: thu được nghiệm $x=x_{i+1}$, dừng chương trình
    - Kiểm tra tại lần lặp đầu tiên:
        - Nếu $f(a) \cdot f(x_{1})<0$: cố định $d=a$ cho các lần lặp tiếp theo
        - Nếu $f(a) \cdot f(x_{1})>0$: cố định $d=b$ cho các lần lặp tiếp theo
    - Cập nhật $i=i+1$
- Bước 5: Sau khi thoả mãn điều kiện dừng với $n$ lần lặp, thu được nghiệm gần đúng là $x=\dfrac{d \cdot f(x_{n-1}) - x_{n} \cdot f(d)}{f(x_{n})-f(d)}$

### 1.2 Code
#### 1.2.1 Có cộng thêm sai số làm tròn
##### Công thức 1

In [2]:
def secant_iteration_v1 (f, df, a, b, n, rbl):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval");
		return 0

	# Implementing Secant Method
	x = a
	mrk = b

	results = []
	for i in range(n):
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		delta_x = abs(f(x_new))/ m1

		results.append({
            'n': i,
			'mrk': mrk,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=|f(x_n)| / m_1': delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b

	# Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl) #must calculate roundoff error
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [3]:
f = lambda x: x ** 5 - 12; 
df = lambda x: 5 * (x ** 4)

a = 1
b = 2

n = 20
rbl = 9
secant_iteration_v1 (f, df, a, b, n, rbl)

m1 = 5, M1 = 80
 n  mrk           x_(n+1)         f(x_(n+1))  delta_x=|f(x_n)| / m_1
 0    2 1.354838709677419 -7.435029421585015       1.487005884317003
 1    2 1.529680628069611 -3.624633240286681       0.724926648057336
 2    2 1.601839853218664 -1.453812878149883       0.290762575629977
 3    2 1.628821087381766 -0.535188563101631       0.107037712620326
 4    2 1.638494760856317 -0.190669278217582       0.038133855643516
 5    2 1.641908612178906 -0.067130140804897       0.013426028160979
 6    2 1.643106527631529 -0.023536232519628       0.004707246503926
 7    2 1.643526030343380 -0.008239825857270       0.001647965171454
 8    2 1.643672834033169 -0.002883205004572       0.000576641000914
 9    2 1.643724194842391 -0.001008683129713       0.000201736625943
10    2 1.643742162405922 -0.000352863397884       0.000070572679577
11    2 1.643748447812582 -0.000123438003291       0.000024687600658
12    2 1.643750646548026 -0.000043180514584       0.000008636102917
13    2 1.64375141

#### Công thức 2

In [4]:
def secant_iteration_v2 (f, df, a, b, n, rbl):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval");
		return 0;

	# Implementing Secant Method
	x = a; mrk = b;

	results = [];
	for i in range(n):
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		delta_x = (M1 - m1) * abs(x - x_new) / m1

		results.append({
            'n': i,
			'mrk': mrk,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=(M1 - m1)*(x-x_new)/m1': delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b

	# Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl) #must calculate roundoff error
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [5]:
f = lambda x: x ** 5 - 12; 
df = lambda x: 5 * (x ** 4)

a = 1
b = 2

n = 20
rbl = 9
secant_iteration_v2 (f, df, a, b, n, rbl)

m1 = 5, M1 = 80
 n  mrk           x_(n+1)         f(x_(n+1))  delta_x=(M1 - m1)*(x-x_new)/m1
 0    2 1.354838709677419 -7.435029421585015               5.322580645161289
 1    2 1.529680628069611 -3.624633240286681               2.622628775882871
 2    2 1.601839853218664 -1.453812878149883               1.082388377235792
 3    2 1.628821087381766 -0.535188563101631               0.404718512446542
 4    2 1.638494760856317 -0.190669278217582               0.145105102118266
 5    2 1.641908612178906 -0.067130140804897               0.051207769838835
 6    2 1.643106527631529 -0.023536232519628               0.017968731789334
 7    2 1.643526030343380 -0.008239825857270               0.006292540677775
 8    2 1.643672834033169 -0.002883205004572               0.002202055346837
 9    2 1.643724194842391 -0.001008683129713               0.000770412138328
10    2 1.643742162405922 -0.000352863397884               0.000269513452962
11    2 1.643748447812582 -0.000123438003291               0

### 1.2.2 Không cộng thêm sai số làm tròn

Đánh giá sai số theo 1 trong 2 công thức:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1} < \epsilon$ $\quad\quad(1)$

+ $|x_n - x^*| \leqslant \dfrac{M_1 - m_1}{m_1} \cdot |x_n - x_{n-1}| < \epsilon$ $\quad\quad(2)$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_1 = \max\limits_{[a,b]} |f'(x)|$

#### Công thức 1

Từ công thức (1) suy ra Điều kiện dừng: $|f(x_n)| < m_1 \varepsilon $

In [6]:
def secant_recursion_absolute_v1 (f, df, a, b, eps):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval")
		return 0

	# Implementing Secant Method
	x = a
	mrk = b 
	new_eps = m1*eps
	print(f"delta_x = {new_eps}")

	results = [] 
	i = 0
	while True:
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		current_delta_x = abs(f(x_new))

		results.append({
            'n': i,
			'mrk': mrk,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=|f(x_n)|': current_delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b
				
        # Stopping condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

    # Print the final result
	df_results = pd.DataFrame(results) 
	print(df_results.to_string(index=False))

	print(f"The value of root with absolute error {eps} is: {x}")

Bài tập VD trên lớp:

In [7]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

secant_recursion_absolute_v1 (f, df, a, b, eps)

m1 = 1.5476066863791802, M1 = 1.945202904840245
delta_x = 7.738033431895901e-08
 n  mrk            x_(n+1)         f(x_(n+1))  delta_x=|f(x_n)|
 0   -2 -2.572246866580031 -0.047010325486338 0.047010325486338
 1   -3 -2.596955165330622 -0.001361749350662 0.001361749350662
 2   -3 -2.597669622547500 -0.000038630153393 0.000038630153393
 3   -3 -2.597689889273960 -0.000001095197097 0.000001095197097
 4   -3 -2.597690463851753 -0.000000031049218 0.000000031049218
The value of root with absolute error 5e-08 is: -2.597690463851753


Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

In [8]:
f = lambda x: np.log(x)-1
df = lambda x: 1/x

a = 2
b = 3
eps = 0.5 * pow(10, -7)
secant_recursion_absolute_v1 (f, df, a, b, eps)

m1 = 0.3333333333333333, M1 = 0.5
delta_x = 1.6666666666666664e-08
 n  mrk           x_(n+1)        f(x_(n+1))  delta_x=|f(x_n)|
 0    3 2.756792171024977 0.014067746909731 0.014067746909731
 1    2 2.723617728992985 0.001961044002238 0.001961044002238
 2    2 2.719022578401181 0.000272469551561 0.000272469551561
 3    2 2.718384689674588 0.000037839810558 0.000037839810558
 4    2 2.718296112392272 0.000005254751567 0.000005254751567
 5    2 2.718283812023015 0.000000729712138 0.000000729712138
 6    2 2.718282103910405 0.000000101332887 0.000000101332887
 7    2 2.718281866710123 0.000000014071785 0.000000014071785
The value of root with absolute error 5e-08 is: 2.718281866710123


Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

In [9]:
f = lambda x: np.tan(x/4)-1
df = lambda x: 1/(4*(np.cos(x/4))**2)

a = 3
b = 3.2
eps = 0.5 * pow(10, -7)
secant_recursion_absolute_v1 (f, df, a, b, eps)

m1 = 0.4669679910450819, M1 = 0.515038889541189
delta_x = 2.3348399552254093e-08
 n  mrk           x_(n+1)         f(x_(n+1))  delta_x=|f(x_n)|
 0  3.2 3.139539120591776 -0.001026239734878 0.001026239734878
 1  3.2 3.141562527343464 -0.000015063009717 0.000015063009717
 2  3.2 3.141592211551216 -0.000000221019264 0.000000221019264
 3  3.2 3.141592647103802 -0.000000003242996 0.000000003242996
The value of root with absolute error 5e-08 is: 3.141592647103802


Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

In [10]:
f = lambda x: x**5-17
df = lambda x: 5*x**4

a = 1
b = 2
eps = 0.5 * pow(10, -5)
secant_recursion_absolute_v1 (f, df, a, b, eps)

m1 = 5, M1 = 80
delta_x = 2.5e-05
 n  mrk           x_(n+1)         f(x_(n+1))  delta_x=|f(x_n)|
 0    2 1.516129032258065 -8.989109037847472 8.989109037847472
 1    2 1.697443347951021 -2.907876413733710 2.907876413733710
 2    2 1.746572420096993 -0.747020411714363 0.747020411714363
 3    2 1.758594730993223 -0.179889751924819 0.179889751924819
 4    2 1.761455511582849 -0.042633986796407 0.042633986796407
 5    2 1.762131596806917 -0.010065952429237 0.010065952429237
 6    2 1.762291114562438 -0.002374452888258 0.002374452888258
 7    2 1.762328737176869 -0.000559989851265 0.000559989851265
 8    2 1.762337609745307 -0.000132061138423 0.000132061138423
 9    2 1.762339702124607 -0.000031143311904 0.000031143311904
10    2 1.762340195558835 -0.000007344349665 0.000007344349665
The value of root with absolute error 5e-06 is: 1.762340195558835


Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp dây cung.

In [11]:
f = lambda x: np.exp(x)-np.cos(2*x)
df = lambda x: np.exp(x)+2*np.sin(2*x)

a = -0.6
b = -0.3
eps = 0.5 * pow(10, -8)
secant_recursion_absolute_v1 (f, df, a, b, eps)

m1 = 0.38846672610835287, M1 = 1.315266535840426
delta_x = 1.9423336305417645e-09
 n  mrk            x_(n+1)         f(x_(n+1))  delta_x=|f(x_n)|
 0 -0.3 -0.393571608980660 -0.031228759902019 0.031228759902019
 1 -0.6 -0.423185833684605 -0.007747782099022 0.007747782099022
 2 -0.6 -0.430239932019379 -0.001720696181665 0.001720696181665
 3 -0.6 -0.431792243198658 -0.000372537831818 0.000372537831818
 4 -0.6 -0.432127654824050 -0.000080208811920 0.000080208811920
 5 -0.6 -0.432199839170246 -0.000017248565964 0.000017248565964
 6 -0.6 -0.432215360672904 -0.000003708274432 0.000003708274432
 7 -0.6 -0.432218697579106 -0.000000797198826 0.000000797198826
 8 -0.6 -0.432219414938725 -0.000000171378468 0.000000171378468
 9 -0.6 -0.432219569153553 -0.000000036842131 0.000000036842131
10 -0.6 -0.432219602305919 -0.000000007920143 0.000000007920143
11 -0.6 -0.432219609432854 -0.000000001702633 0.000000001702633
The value of root with absolute error 5e-09 is: -0.43221960943285404


#### Công thức 2

Từ công thức (2) suy ra Điều kiện dừng: $|x_n - x_{n-1}| < \epsilon \dfrac{m_1}{M_1 - m_1}$

In [12]:
def secant_recursion_absolute_v2 (f, df, a, b, eps):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval")
		return 0

	# Implementing Secant Method
	x = a; mrk = b; new_eps = eps*m1/(M1 - m1)
	print(f"delta_x = {new_eps}")

	results = []; i = 0; 
	while True:
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		current_delta_x = abs(x - x_new)

		results.append({
            'n': i,
			'mrk': mrk,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=|x_(n+1) - x_n|': current_delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b
				
        # Stopping condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

    # Print the final result
	df_results = pd.DataFrame(results) 
	print(df_results.to_string(index=False))

	print(f"The value of root with absoulte error {eps} is: {x}")

Bài tập VD trên lớp:

In [13]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

secant_recursion_absolute_v2 (f, df, a, b, eps)

m1 = 1.5476066863791802, M1 = 1.945202904840245
delta_x = 1.9462039809751506e-07
 n  mrk            x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0   -2 -2.572246866580031 -0.047010325486338        0.427753133419969
 1   -3 -2.596955165330622 -0.001361749350662        0.024708298750591
 2   -3 -2.597669622547500 -0.000038630153393        0.000714457216878
 3   -3 -2.597689889273960 -0.000001095197097        0.000020266726460
 4   -3 -2.597690463851753 -0.000000031049218        0.000000574577793
 5   -3 -2.597690480141232 -0.000000000880255        0.000000016289479
The value of root with absoulte error 5e-08 is: -2.597690480141232


Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

In [14]:
f = lambda x: np.log(x)-1
df = lambda x: 1/x

a = 2
b = 3
eps = 0.5 * pow(10, -7)
secant_recursion_absolute_v2 (f, df, a, b, eps)

m1 = 0.3333333333333333, M1 = 0.5
delta_x = 9.999999999999997e-08
 n  mrk           x_(n+1)        f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0    3 2.756792171024977 0.014067746909731        0.756792171024977
 1    2 2.723617728992985 0.001961044002238        0.033174442031992
 2    2 2.719022578401181 0.000272469551561        0.004595150591805
 3    2 2.718384689674588 0.000037839810558        0.000637888726592
 4    2 2.718296112392272 0.000005254751567        0.000088577282317
 5    2 2.718283812023015 0.000000729712138        0.000012300369256
 6    2 2.718282103910405 0.000000101332887        0.000001708112610
 7    2 2.718281866710123 0.000000014071785        0.000000237200283
 8    2 2.718281833770854 0.000000001954105        0.000000032939269
The value of root with absoulte error 5e-08 is: 2.718281833770854


Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

In [15]:
f = lambda x: np.tan(x/4)-1
df = lambda x: 1/(4*(np.cos(x/4))**2)

a = 3
b = 3.2
eps = 0.5 * pow(10, -7)
secant_recursion_absolute_v2 (f, df, a, b, eps)

m1 = 0.4669679910450819, M1 = 0.515038889541189
delta_x = 4.857075753253268e-07
 n  mrk           x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0  3.2 3.139539120591776 -0.001026239734878        0.139539120591776
 1  3.2 3.141562527343464 -0.000015063009717        0.002023406751687
 2  3.2 3.141592211551216 -0.000000221019264        0.000029684207752
 3  3.2 3.141592647103802 -0.000000003242996        0.000000435552586
The value of root with absoulte error 5e-08 is: 3.141592647103802


Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

In [16]:
f = lambda x: x**5-17
df = lambda x: 5*x**4

a = 1
b = 2
eps = 0.5 * pow(10, -5)
secant_recursion_absolute_v2 (f, df, a, b, eps)

m1 = 5, M1 = 80
delta_x = 3.3333333333333335e-07
 n  mrk           x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0    2 1.516129032258065 -8.989109037847472        0.516129032258065
 1    2 1.697443347951021 -2.907876413733710        0.181314315692957
 2    2 1.746572420096993 -0.747020411714363        0.049129072145971
 3    2 1.758594730993223 -0.179889751924819        0.012022310896231
 4    2 1.761455511582849 -0.042633986796407        0.002860780589626
 5    2 1.762131596806917 -0.010065952429237        0.000676085224068
 6    2 1.762291114562438 -0.002374452888258        0.000159517755521
 7    2 1.762328737176869 -0.000559989851265        0.000037622614431
 8    2 1.762337609745307 -0.000132061138423        0.000008872568438
 9    2 1.762339702124607 -0.000031143311904        0.000002092379300
10    2 1.762340195558835 -0.000007344349665        0.000000493434228
11    2 1.762340311922558 -0.000001731974969        0.000000116363723
The value of root with absoulte error 5e-

Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp dây cung.

In [17]:
f = lambda x: np.exp(x)-np.cos(2*x)
df = lambda x: np.exp(x)+2*np.sin(2*x)

a = -0.6
b = -0.3
eps = 0.5 * pow(10, -8)
secant_recursion_absolute_v2 (f, df, a, b, eps)

m1 = 0.38846672610835287, M1 = 1.315266535840426
delta_x = 2.0957423708397935e-09
 n  mrk            x_(n+1)         f(x_(n+1))  delta_x=|x_(n+1) - x_n|
 0 -0.3 -0.393571608980660 -0.031228759902019        0.206428391019340
 1 -0.6 -0.423185833684605 -0.007747782099022        0.029614224703945
 2 -0.6 -0.430239932019379 -0.001720696181665        0.007054098334774
 3 -0.6 -0.431792243198658 -0.000372537831818        0.001552311179279
 4 -0.6 -0.432127654824050 -0.000080208811920        0.000335411625393
 5 -0.6 -0.432199839170246 -0.000017248565964        0.000072184346195
 6 -0.6 -0.432215360672904 -0.000003708274432        0.000015521502658
 7 -0.6 -0.432218697579106 -0.000000797198826        0.000003336906202
 8 -0.6 -0.432219414938725 -0.000000171378468        0.000000717359619
 9 -0.6 -0.432219569153553 -0.000000036842131        0.000000154214827
10 -0.6 -0.432219602305919 -0.000000007920143        0.000000033152367
11 -0.6 -0.432219609432854 -0.000000001702633        0.00000000712

### 1.2.3 Tìm nghiệm thoả mãn điều kiện sai số tương đối

Đánh giá sai số theo công thức: $\dfrac{|x_n - x^*|}{|x_n|} \leqslant \dfrac{M_1 - m_1}{m_1} \cdot \dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_1 = \max\limits_{[a,b]} |f'(x)|$


Từ công thức sai số hậu nghiệm, ta có Điều kiện dừng:  $\dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta \dfrac{m_1}{M_1 - m_1}$

In [18]:
def secant_recursion_relative (f, df, a, b, eta):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval");
		return 0;

	# Implementing Secant Method
	x = a; mrk = b; new_eta = eta * m1/(M1 - m1);
	print(f"sigma_x = {new_eta}, m_1 = {m1}, M_1 = {M1}")

	results = []; i = 0;
	while True:
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		current_sigma_x = abs(x - x_new)/abs(x_new)

		results.append({
            'n': i,
			'mrk': mrk,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'sigma_x=|x_(n+1)-x_n|/|x_n|': current_sigma_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b
				
        # Stopping condition
		if current_sigma_x < new_eta:
			break
		else:
			i += 1

    # Print the final result
	df_results = pd.DataFrame(results) 
	print(df_results.to_string(index=False))

	print(f"The value of root with relative error {eta} is: {x}")

Câu 11. Giải phương trình với nghiệm có sai lệch không vượt quá 0.05% bằng phương pháp dây cung.

a. $x^5-3x^3+2x^2-x+5=0$

In [19]:
f = lambda x: x**5 - 3*x**3 + 2*x**2 - x + 5
df = lambda x: 5*x**4 - 9*x**2 + 4*x - 1

a = -3
b = -2

eta = 0.05 * pow(10, -2)

secant_recursion_relative (f, df, a, b, eta)

m1 = 35, M1 = 311
sigma_x = 6.340579710144928e-05, m_1 = 35, M_1 = 311
 n  mrk            x_(n+1)        f(x_(n+1))  sigma_x=|x_(n+1)-x_n|/|x_n|
 0   -2 -2.048951048951049 5.138543296012815            0.464163822525597
 1   -3 -2.083576645173499 3.633621955959093            0.016618345335488
 2   -3 -2.107424311490581 2.500536154289734            0.011316025058197
 3   -3 -2.123539178923813 1.688293466358496            0.007588683831771
 4   -3 -2.134286084419477 1.125129162756630            0.005035363147479
 5   -3 -2.141389377440757 0.743280711796173            0.003317142176996
 6   -3 -2.146056434654608 0.488176034016396            0.002174713180179
 7   -3 -2.149110726939240 0.319399617813440            0.001421188888198
 8   -3 -2.151104380882693 0.208449064921011            0.000926804836237
 9   -3 -2.152403503655438 0.135816173304101            0.000603568415745
10   -3 -2.153249110020282 0.088396961042743            0.000392711814397
11   -3 -2.153799121682598 0.057493650009

b. $2^x-5x+\sin x = 0$

In [20]:
f = lambda x: 2**x - 5*x + np.sin(x)
df = lambda x: 2**x*np.log(2) - 5 + np.cos(x)

a = 0
b = 1

eta = 0.05 * pow(10, -2)

secant_recursion_relative (f, df, a, b, eta)

m1 = 3.0734033330119694, M1 = 3.3068528194400546
sigma_x = 0.006582587479708893, m_1 = 3.0734033330119694, M_1 = 3.3068528194400546
 n  mrk           x_(n+1)         f(x_(n+1))  sigma_x=|x_(n+1)-x_n|/|x_n|
 0    1 0.316603075415845 -0.026280379774505            1.000000000000000
 1    0 0.308495691484825 -0.000437047622463            0.026280379774506
 2    0 0.308360923076534 -0.000007294904439            0.000437047622463
The value of root with relative error 0.0005 is: 0.308360923076534


c. $e^{-x}-x=0$

In [21]:
f = lambda x: np.e**(-x) - x
df = lambda x: -np.e**(-x) - 1

a = 0
b = 1

eta = 0.05 * pow(10, -2)

secant_recursion_relative (f, df, a, b, eta)

m1 = 1.3678794411714423, M1 = 2.0
sigma_x = 0.0010819767068693264, m_1 = 1.3678794411714423, M_1 = 2.0
 n  mrk           x_(n+1)         f(x_(n+1))  sigma_x=|x_(n+1)-x_n|/|x_n|
 0    1 0.612699836780282 -0.070813947873171            1.000000000000000
 1    0 0.572181412090508 -0.007888272855300            0.070813947873171
 2    0 0.567703214235785 -0.000877391979772            0.007888272855300
 3    0 0.567205552633023 -0.000097572726128            0.000877391979772
The value of root with relative error 0.0005 is: 0.5672055526330225
